In [1]:
import sys
from pathlib import Path

root = Path().resolve()
sys.path.insert(0, str(root / "src"))


In [2]:
from FLRW_Net.networks.test import GeneralizedModel
import tensorflow as tf

tf.keras.backend.set_floatx("float64")

In [3]:
from FLRW_Net.utils.eoms import eom_struts

flrw_net = GeneralizedModel(number_of_timesteps=1, triangulation="5-cell", cosmological_constant=1e-6)

prediction = tf.constant([[-1, 0.7, -2]], dtype=tf.float64)
print(eom_struts(prediction, flrw_net.model_params))

ValueError: numerator1: 5.249999999999999e-05, numerator2: -31.646196176510042, denominator1: 11.509995655950522, denominator2: 0.9797958971132711, arg: -0.021276595744681

In [3]:
def crop(arg):
    """Set the argument range of 'tf.acos' to [-1., 1.] since numeric computations could
    lead to values that are slightly outside this range due to computational error.
    """
    minus_one = tf.constant(-1, dtype=tf.float64)
    one = tf.constant(1, dtype=tf.float64)
    # Define the conditions
    too_small = arg < minus_one
    too_large = arg > one

    # Use tensorflow's if/then/else: (if condition, then, else(if condition2, then, else))
    output = tf.where(too_small, minus_one, tf.where(too_large, one, arg))

    return output

In [5]:
import numpy as np
def custom_compute_loss(prediction, model_params):
    """Compute the network's loss by evaluating the EOMm term, which should be 0 in the
    presence of a classical solution. Thus, its value represents a natural choice for the loss.
    """
    # Define the parts of the output in Regge calculus variables
    l1 = prediction[0, 0]
    m1 = prediction[0, 1]
    l2 = prediction[0, 2]

    lamb = model_params.lamb
    n3 = model_params.n3
    n1 = model_params.n1
    nte = model_params.nte

    pi = tf.constant(np.pi, dtype=tf.float64)

    # Compute the value of the EOMm: the first fraction
    numerator1 = -(l1 + l2) * (tf.math.square(l1) + tf.math.square(l2)) * lamb * m1 * n3
    denominator1 = tf.constant(12, dtype=tf.float64) * tf.math.sqrt(
        tf.constant(-3, dtype=tf.float64) * tf.math.square(l1 - l2) + tf.constant(8, dtype=tf.float64) * tf.math.square(m1)
    )

    # Compute the value of the EOMm: the second fraction
    # 'crop' ensures the arguments of the arccos to be indeed in [-1, 1]
    arg = crop(
        (tf.math.square(l1 - l2) - tf.constant(2, dtype=tf.float64) * tf.math.square(m1))
        / (tf.constant(2, dtype=tf.float64) * tf.math.square(l1 - l2) - tf.constant(6, dtype=tf.float64) * tf.math.square(m1))
    )
    numerator2 = (l1 + l2) * m1 * n1 * (tf.constant(2, dtype=tf.float64) * pi - nte * tf.math.acos(arg))
    denominator2 = tf.math.sqrt(
        -tf.constant(1, dtype=tf.float64) * tf.math.square(l1 - l2) + tf.constant(4, dtype=tf.float64) * tf.math.square(m1)
    )

    # Define the total loss as: EOMm ^ 2. This ensures that the solution of the time step
    # can be found as a minimizing procedure, since loss = 0 <--> EOMm = 0 i.e.,
    # a classical solution
    loss = tf.math.square(numerator1 / denominator1 + numerator2 / denominator2)
    print(f"numerator1: {numerator1}, numerator2: {numerator2}, denominator1: {denominator1}, denominator2: {denominator2}, arg: {arg}")
    return loss

In [14]:
from FLRW_Net.utils.utils import set_model_params
import tensorflow as tf
model_params = set_model_params("600-cell", cosmological_constant=1e-5)
inputs = tf.constant([[100, 70.1, 200]], dtype=tf.float64)
print(custom_compute_loss(inputs, model_params))

numerator1: -6309000.0, numerator2: -25157302.2997475, denominator1: 1157.9894300035726, denominator2: 98.26515150347043, arg: -0.018133584140125794
tf.Tensor(68362755375.17299, shape=(), dtype=float64)


In [29]:
def arg1(inputs):
    """Argument of dihedral angle 1"""
    l1 = inputs[0, 0]
    m1 = inputs[0, 1]
    l2 = inputs[0, 2]
    return crop(
        (tf.math.square(l1 - l2) - tf.constant(2, dtype=tf.float64) * tf.math.square(m1))
        / (tf.constant(2, dtype=tf.float64) * tf.math.square(l1 - l2) - tf.constant(6, dtype=tf.float64) * tf.math.square(m1))
    )

In [4]:
def arg2(inputs):
    """Argument of dihedral angle 2"""
    l1 = inputs[0, 0]
    m1 = inputs[0, 1]
    l2 = inputs[0, 2]
    return crop(
        (-l1 + l2)
        / (
            tf.constant(2.0, dtype=tf.float64)
            * tf.math.sqrt(
                tf.constant(-2.0, dtype=tf.float64) * tf.math.square(l1 - l2) + tf.constant(6.0, dtype=tf.float64) * tf.math.square(m1)
            )
        )
    )

In [79]:
import tensorflow as tf
import numpy as np
inputs = tf.constant([[4, 10, 9]], dtype=tf.float64)
print(arg2(inputs))

tf.Tensor(0.10660035817780521, shape=(), dtype=float64)


In [49]:
import tensorflow as tf
import numpy as np
inputs = tf.constant([[1, 0.68, 2]], dtype=tf.float64)
print(arg1(inputs))

tf.Tensor(-0.09710743801652864, shape=(), dtype=float64)


In [40]:
def EOMl1(inputs, params):
    """Part of the equation of motion of the spatial edge that has the number of tetrahedra n3 as a prefactor."""
    l1 = inputs[0, 0]
    m1 = inputs[0, 1]
    l2 = inputs[0, 2]
    output = (
        1
        / 24
        * params.n3
        * params.lamb
        * (3 * tf.math.pow(l2, 3) * (-l1 + l2) - 2 * (tf.math.square(l1 + l2) + 2 * tf.math.square(l2)) * tf.math.square(m1))
        / tf.math.sqrt(-3 * tf.math.square(l1 - l2) + 8 * tf.math.square(m1))
    )
    return output

In [105]:
import tensorflow as tf
import numpy as np
inputs = tf.constant([[1, 0.66, 2]], dtype=tf.float64)
flrw_net = GeneralizedModel(1, triangulation="5-cell", cosmological_constant=1e-5)
print(EOMl1(inputs, flrw_net.model_params))

tf.Tensor(2.749628781575549e-05, shape=(), dtype=float64)


In [ ]:
def EOMl_acoses(inputs, params):
    pi = tf.constant(np.pi, dtype=tf.float64)
    """Part of the equation of motion of the spatial edge that has the number of triangles n2 as a prefactor."""
    l2 = inputs[0, 2]
    tmp1 = crop(arg2(inputs[:, :3]))
    tmp2 = crop(arg2(tf.reverse(inputs[:, 2:], axis=[1])))
    output = tf.math.sqrt(tf.constant(3.0, dtype=tf.float64)) * l2 * params.n2 * (pi - tf.math.acos(tmp1) - tf.math.acos(tmp2))
    return output

In [23]:
import tensorflow as tf
import numpy as np

inputs = tf.constant([[4, 10, 7, 20, 12]], dtype=tf.float64)
flrw_net = GeneralizedModel(1, triangulation="600-cell", cosmological_constant=1e-5)
print(EOMl_acoses(inputs, flrw_net.model_params))

tf.Tensor([[12. 20.  7.]], shape=(1, 3), dtype=float64)
tf.Tensor(154.55910664618892, shape=(), dtype=float64)


In [24]:
def EOMl2(inputs, params):
    """Part of the equation of motion of the spatial edge that has the number of edges n1 as a prefactor."""
    l1 = inputs[0, 0]
    m1 = inputs[0, 1]
    l2 = inputs[0, 2]
    m2 = inputs[0, 3]
    l3 = inputs[0, 4]
    pi = tf.constant(np.pi, dtype=tf.float64)
    output = (
        params.n1
        * (2 * pi - params.nte * tf.math.acos(crop(arg1(inputs[:, :3]))))
        * tf.math.sqrt(-tf.math.square(l2 - l3) + 4 * tf.math.square(m2))
        * (-2 * tf.math.square(m1) - l1 * l2 + tf.math.square(l2))
        / (
            2
            * tf.math.sqrt(-tf.math.square(l1 - l2) + 4 * tf.math.square(m1))
            * tf.math.sqrt(-tf.math.square(l2 - l3) + 4 * tf.math.square(m2))
        )
    )
    return output

In [38]:
import tensorflow as tf
import numpy as np

inputs = tf.constant([[4, 0.66, 7, 0.68, 12]], dtype=tf.float64)
flrw_net = GeneralizedModel(1, triangulation="5-cell", cosmological_constant=1e-5)
print(EOMl2(inputs, flrw_net.model_params))

tf.Tensor(nan, shape=(), dtype=float64)


In [39]:
def EOM_edges(inputs, params):
    """Final EOMl term ^ 2"""
    term1 = EOMl1(inputs[:, :3], params)
    term2 = EOMl1(tf.reverse(inputs[:, 2:], axis=[1]), params)
    term3 = EOMl_acoses(inputs, params)
    term4 = EOMl2(inputs, params)
    term5 = EOMl2(tf.reverse(inputs, axis=[1]), params)
    return tf.math.square(term1 + term2 + term3 - term4 - term5)

In [50]:
import tensorflow as tf
import numpy as np

inputs = tf.constant([[4, 10, 7, 20, 12]], dtype=tf.float64)
flrw_net = GeneralizedModel(1, triangulation="600-cell", cosmological_constant=1e-5)
print(EOM_edges(inputs, flrw_net.model_params))

tf.Tensor([[12. 20.  7.]], shape=(1, 3), dtype=float64)
tf.Tensor(1721454.1455350781, shape=(), dtype=float64)


In [ ]:
from FLRW_Net.utils.eoms import eom_struts

prediction = tf.constant([[1, 0.66, 2, 0.67, 3]], dtype=tf.float64)

slices = tf.signal.frame(
    prediction,
    frame_length=3,
    frame_step=2,
    axis=1
)[:, 0:2]
print(slices)
def fn(slice_):
    return eom_struts(slice_, flrw_net.model_params)
slices = tf.transpose(slices, perm=[1, 0, 2])
losses = tf.map_fn(
    fn,
    slices,
    fn_output_signature=tf.float64
)
print(tf.expand_dims(losses, axis=0))

In [ ]:
inputs = tf.constant([[1, 0.66, 2, 0.67, 3, 0.68, 4]], dtype=tf.float64)
number_of_timesteps = int((tf.shape(inputs)[1]-tf.constant(1))/tf.constant(2))
print(number_of_timesteps)

flrw_net = GeneralizedModel(number_of_timesteps, triangulation="5-cell", cosmological_constant=1e-6)
adam_optimizer = tf.keras.optimizers.Adam(
    learning_rate=1e-5,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-7,
    amsgrad=False,
    decay=0.9,
    clipnorm=2,
    clipvalue=None,
    global_clipnorm=None,
)
flrw_net.compile(optimizer=adam_optimizer)

flrw_net.training(inputs=inputs, epochs=10)

In [ ]:
inputs2 = tf.constant([[1, 0.66, 2, 0.67, 3, 0.68, 4, 0.69, 5, 0.7, 6, 0.71, 7]], dtype=tf.float64)
number_of_timesteps2 = int((tf.shape(inputs)[1]-tf.constant(1))/tf.constant(2))
print(number_of_timesteps2)

flrw_net2 = GeneralizedModel(number_of_timesteps2, triangulation="600-cell", cosmological_constant=1e-6)
flrw_net2.compile(optimizer=adam_optimizer)

flrw_net2.training(inputs=inputs2, epochs=1000)